# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import col
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 1a72e667-8217-4a2e-af10-9c80b2bfb8ea
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 1a72e667-8217-4a2e-af10-9c80b2bfb8ea to get into ready status...
Session 1a72e667-8217-4a2e-af10-9c80b2bfb8ea 

### Leitura das bases da camada Silver


In [14]:
path_fato = "s3://datalake-404604958697/silver/pnad_covid/base_tratada/"
path_dic = "s3://datalake-404604958697/silver/dicionario_covid/"

df_fato = spark.read.parquet(path_fato)
df_dic = spark.read.parquet(path_dic)

#### Criando dicionários filtrados


In [15]:
# Características --------------------

dic_sexo = df_dic.filter(col("cod_variavel") == "A003")
dic_urbano_rural = df_dic.filter(col("cod_variavel") == "V1022")
dic_parentesco = df_dic.filter(col("cod_variavel") == "A001A")
dic_uf = df_dic.filter(col("cod_variavel") == "UF")
dic_capital = df_dic.filter(col("cod_variavel") == "CAPITAL")

# Sintomas --------------------

dic_febre = df_dic.filter(col("cod_variavel") == "B0011")
dic_tosse = df_dic.filter(col("cod_variavel") == "B0012")
dic_dificuldade_respiratoria = df_dic.filter(col("cod_variavel") == "B0014")
dic_dor_cabeca = df_dic.filter(col("cod_variavel") == "B0015")
dic_perda_olfato_paladar = df_dic.filter(col("cod_variavel") == "B00111")

# Comportamento --------------------

dic_atendimento_sus = df_dic.filter(col("cod_variavel") == "B0043")
dic_atendimento_privado = df_dic.filter(col("cod_variavel") == "B0045")
dic_ficou_internado = df_dic.filter(col("cod_variavel") == "B005")
dic_ficou_entubado = df_dic.filter(col("cod_variavel") == "B006")
dic_ficou_em_casa = df_dic.filter(col("cod_variavel") == "B0031")

# Econômicas --------------------

dic_se_afastou = df_dic.filter(col("cod_variavel") == "C002")
dic_motivo_afastamento = df_dic.filter(col("cod_variavel") == "C003")
dic_cargo = df_dic.filter(col("cod_variavel") == "C007C")
dic_ramo = df_dic.filter(col("cod_variavel") == "C007D")
dic_faixa_renda = df_dic.filter(col("cod_variavel") == "C01011")

### Joins


In [16]:
df_dic.cache()

DataFrame[cod_variavel: string, descricao_variavel: string, cod_categoria: string, descricao_categoria: string, id_categoria: string]


Corrigindo valores nulls

In [17]:
from pyspark.sql.functions import coalesce, lit

cols = [
    "dificuldade_respiratoria",
    "dor_de_cabeca",
    "perda_olfato_paladar",
    "buscou_atendimento_sus",
    "buscou_atendimento_privado",
    "ficou_internado",
    "ficou_entubado",
    "ficou_em_casa",
    "se_afastou",
    "motivo_afastamento",
    "cargo",
    "ramo",
    "faixa_renda"
]

for c in cols:
    df_fato = df_fato.withColumn(c, coalesce(col(c), lit("99")))

#### Caracteristicas

In [18]:
# Caracteristicas -----------------------------

df_gold = df_fato \
.join(dic_sexo, df_fato.sexo == dic_sexo.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","sexo_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_urbano_rural, df_gold.urbano_rural == dic_urbano_rural.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","urbano_rural_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_parentesco, df_gold.parentesco == dic_parentesco.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","parentesco_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_uf, df_gold.uf == dic_uf.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","uf_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_capital, df_gold.capital == dic_capital.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","capital_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

# Sintomas -----------------------------

df_gold = df_gold \
.join(dic_febre, df_gold.febre == dic_febre.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","febre_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_tosse, df_gold.tosse == dic_tosse.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","tosse_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_dificuldade_respiratoria,df_gold.dificuldade_respiratoria == dic_dificuldade_respiratoria.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","dificuldade_respiratoria_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_dor_cabeca, df_gold.dor_de_cabeca == dic_dor_cabeca.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","dor_cabeca_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_perda_olfato_paladar,
      df_gold.perda_olfato_paladar == dic_perda_olfato_paladar.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","perda_olfato_paladar_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

# Comportamento -----------------------------

df_gold = df_gold \
.join(dic_atendimento_sus, df_gold.buscou_atendimento_sus == dic_atendimento_sus.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","buscou_atendimento_sus_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_atendimento_privado, df_gold.buscou_atendimento_privado == dic_atendimento_privado.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","buscou_atendimento_privado_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_ficou_internado, df_gold.ficou_internado == dic_ficou_internado.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","ficou_internado_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_ficou_entubado, df_gold.ficou_entubado == dic_ficou_entubado.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","ficou_entubado_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_ficou_em_casa, df_gold.ficou_em_casa == dic_ficou_em_casa.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","ficou_em_casa_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

# Economicas -----------------------------

df_gold = df_gold \
.join(dic_se_afastou, df_gold.se_afastou == dic_se_afastou.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","se_afastou_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_motivo_afastamento, df_gold.motivo_afastamento == dic_motivo_afastamento.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","motivo_afastamento_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_cargo, df_gold.cargo == dic_cargo.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","cargo_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_ramo, df_gold.ramo == dic_ramo.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","ramo_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

df_gold = df_gold \
.join(dic_faixa_renda, df_gold.faixa_renda == dic_faixa_renda.cod_categoria,"left") \
.withColumnRenamed("descricao_categoria","faixa_renda_desc") \
.drop("cod_categoria","cod_variavel","descricao_variavel","id_categoria")

In [19]:
df_gold.show(10)
df_gold.printSchema()


+-----+----+------------+---+-------+----------+-----+-----+------------------------+-------------+--------------------+----------------------+--------------------------+---------------+--------------+-------------+----------+------------------+-----+----+-----------+----+---+---------+-----------------+--------------------+--------+--------------------+----------+----------+-----------------------------+---------------+-------------------------+---------------------------+-------------------------------+--------------------+-------------------+------------------+---------------+-----------------------+--------------------+--------------------+----------------+
|idade|sexo|urbano_rural| uf|capital|parentesco|febre|tosse|dificuldade_respiratoria|dor_de_cabeca|perda_olfato_paladar|buscou_atendimento_sus|buscou_atendimento_privado|ficou_internado|ficou_entubado|ficou_em_casa|se_afastou|motivo_afastamento|cargo|ramo|faixa_renda| ano|mes|sexo_desc|urbano_rural_desc|     parentesco_desc| uf_

In [20]:
df_gold = df_gold.drop(
    "sexo",
    "urbano_rural",
    "parentesco",
    "uf",
    "capital",
    "febre",
    "tosse",
    "dificuldade_respiratoria",
    "dor_de_cabeca",
    "perda_olfato_paladar",
    "buscou_atendimento_sus",
    "buscou_atendimento_privado",
    "ficou_internado",
    "ficou_entubado",
    "ficou_em_casa",
    "se_afastou",
    "motivo_afastamento",
    "cargo",
    "ramo",
    "faixa_renda"
)
df_gold = df_gold \
.withColumnRenamed("sexo_desc","sexo") \
.withColumnRenamed("febre_desc","febre") \
.withColumnRenamed("tosse_desc","tosse") \
.withColumnRenamed("dificuldade_respiratoria_desc","dificuldade_respiratoria") \
.withColumnRenamed("dor_cabeca_desc","dor_cabeca") \
.withColumnRenamed("perda_olfato_paladar_desc","perda_olfato_paladar") \
.withColumnRenamed("buscou_atendimento_sus_desc","buscou_atendimento_sus") \
.withColumnRenamed("buscou_atendimento_privado_desc","buscou_atendimento_privado") \
.withColumnRenamed("ficou_internado_desc","ficou_internado") \
.withColumnRenamed("ficou_entubado_desc","ficou_entubado") \
.withColumnRenamed("ficou_em_casa_desc","ficou_em_casa") \
.withColumnRenamed("se_afastou_desc","se_afastou") \
.withColumnRenamed("motivo_afastamento_desc","motivo_afastamento") \
.withColumnRenamed("cargo_desc","cargo") \
.withColumnRenamed("ramo_desc","ramo") \
.withColumnRenamed("faixa_renda_desc","faixa_renda") \
.withColumnRenamed("urbano_rural_desc","urbano_rural") \
.withColumnRenamed("parentesco_desc","parentesco") \
.withColumnRenamed("uf_desc","uf") \
.withColumnRenamed("capital_desc","capital")

In [21]:
from pyspark.sql.functions import count

df_gold.groupBy("ano","mes") \
    .agg(count("*").alias("qtd_linhas")) \
    .orderBy("ano","mes") \
    .show(50, False)

+----+---+----------+
|ano |mes|qtd_linhas|
+----+---+----------+
|2020|5  |349306    |
|2020|6  |381270    |
|2020|7  |384166    |
|2020|8  |386520    |
|2020|9  |387298    |
|2020|10 |380461    |
|2020|11 |381438    |
+----+---+----------+


### Salvando na camada Gold

In [22]:
path_gold = "s3://datalake-404604958697/gold/base_analitica_covid/"

df_gold.write \
    .mode("overwrite") \
    .partitionBy("ano","mes") \
    .parquet(path_gold)